# Chapter 3 — Labeling
### AFML by Marcos López de Prado
Real Bitcoin Trade Data · March 2026

This notebook demonstrates two labeling methods from AFML Chapter 3:

- **Section 3.2** — Fixed-Time Horizon: label based on return after N bars
- **Section 3.3-3.5** — Triple Barrier Method: label based on which barrier is hit first

Both methods take CUSUM filter events from Chapter 2 as entry points and assign
each one a label of +1 (win), -1 (loss), or 0 (neutral/expired).

## Setup — Imports

In [ ]:
import sys
import os
%matplotlib inline

# Add both ch02 and ch03 to path so we can import from both
ch02 = os.path.abspath(os.path.join('..', 'ch02'))
ch03 = os.path.abspath('.')
sys.path.insert(0, ch02)
sys.path.insert(0, ch03)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import bars          # from ch02
import labeling      # from ch03

print("Imports successful.")
print(f"ch02 path: {ch02}")
print(f"ch03 path: {ch03}")

## Load Raw Tick Data and Generate Dollar Bars

We use the same Bitcoin tick data from Chapter 2. Dollar bars are the preferred
bar type for labeling — they close based on dollar volume traded, making each
bar represent roughly the same economic activity regardless of price level.

In [ ]:
# Load raw tick data from ch02
data_path = os.path.join(ch02, 'input_data', 'BTCTUSD-trades-2026-03.csv')

raw = pd.read_csv(data_path, header=None,
                  names=['TradeID', 'Price', 'Volume', 'QuoteVolume',
                         'Timestamp', 'IsBuyerMaker', 'IsBestMatch'])

raw['Date']  = pd.to_datetime(raw['Timestamp'], unit='us')
raw['Label'] = raw['IsBuyerMaker'].apply(lambda x: -1 if x else 1)

df = raw[['Date', 'Price', 'Volume', 'Label']].copy()
df['Dollar'] = df['Price'] * df['Volume']
df = bars.delta(df)

print(f"Loaded {len(df):,} ticks")
print(f"Date range: {df['Date'].iloc[0]} to {df['Date'].iloc[-1]}")

In [ ]:
# Generate dollar bars — close every $50,000 of BTC traded
dollar_bars = bars.dollar_bars(df, thresh=50000)

# Set Date as index — labeling functions expect datetime-indexed prices
dollar_bars = dollar_bars.set_index('Date')

print(f"Dollar bars: {len(dollar_bars)} bars")
print(f"Date range: {dollar_bars.index[0]} to {dollar_bars.index[-1]}")
dollar_bars.head()

## Run CUSUM Filter on Dollar Bar Closes

The CUSUM filter from Chapter 2 (Section 2.5) identifies bars where the price
has drifted meaningfully from its last reset level. These become our candidate
trade entry points — the events we will label.

In [ ]:
# Run CUSUM filter on dollar bar close prices
# h=0.02 means fire an event when cumulative return exceeds 2%
# Using percentage threshold since we're working with bar-level returns
close = dollar_bars['Close']

# Build a simple df for CUSUM — it expects a DataFrame with Price and Date columns
cusum_df = pd.DataFrame({
    'Date': close.index,
    'Price': close.values
})

events = bars.cusum_filter(cusum_df, h=500)

print(f"CUSUM filter fired {len(events)} events")
print(f"First 5 events: {events[:5].tolist()}")

## Compute Daily Volatility

Before applying the triple barrier method, we compute daily volatility using
an exponentially weighted moving average of returns (Snippet 3.1).

This volatility estimate is used to set the width of the horizontal barriers
dynamically — so profit targets and stop losses scale with current market
conditions rather than being fixed percentages.

In [ ]:
# Compute daily volatility on dollar bar closes
daily_vol = labeling.get_daily_vol(close, span0=100)

print(f"Daily volatility estimates: {len(daily_vol)} values")
print(f"Mean volatility: {daily_vol.mean():.4f} ({daily_vol.mean()*100:.2f}%)")
print(f"Max volatility:  {daily_vol.max():.4f} ({daily_vol.max()*100:.2f}%)")

# Plot volatility over time
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_vol.index, daily_vol * 100, color='purple', linewidth=0.8)
ax.set_title("Daily Volatility Estimate (EWMA) — Dollar Bar Closes", fontsize=12)
ax.set_xlabel("Date")
ax.set_ylabel("Volatility (%)")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Section 3.2 — Fixed-Time Horizon Labeling

The simplest labeling approach. For each CUSUM event, look h bars into the future
and check whether the return exceeds a threshold τ.

**Limitation:** ignores what happened between entry and exit. A trade that nearly
hit the stop loss before recovering is labeled the same as a smooth upward move.

In [ ]:
# Apply fixed-time horizon labeling
# h=5: look 5 bars ahead
# threshold=0.01: require at least 1% move to get a non-zero label
fth_labels = labeling.fixed_time_horizon(
    close     = close,
    events    = events,
    h         = 5,
    threshold = 0.01
)

print(f"Fixed-time horizon labels: {len(fth_labels)}")
print(f"Label distribution:")
print(fth_labels.value_counts().sort_index())

### Fixed-Time Horizon — Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Fixed-Time Horizon Labeling (h=5 bars, τ=1%)", fontsize=12)

# Bar chart of label counts
counts = fth_labels.value_counts().sort_index()
colors = ['red', 'grey', 'green']
axes[0].bar(['-1 (Loss)', '0 (Neutral)', '+1 (Win)'],
            [counts.get(-1, 0), counts.get(0, 0), counts.get(1, 0)],
            color=colors)
axes[0].set_title("Label Counts")
axes[0].set_ylabel("Number of observations")

# Labels over time
axes[1].scatter(fth_labels.index, fth_labels.values,
                c=fth_labels.map({-1: 'red', 0: 'grey', 1: 'green'}),
                alpha=0.6, s=20)
axes[1].set_title("Labels Over Time")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Label")
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['-1 (Loss)', '0 (Neutral)', '+1 (Win)'])

red_patch   = mpatches.Patch(color='red',   label='-1 Loss')
grey_patch  = mpatches.Patch(color='grey',  label='0 Neutral')
green_patch = mpatches.Patch(color='green', label='+1 Win')
axes[1].legend(handles=[green_patch, grey_patch, red_patch])

plt.tight_layout()
plt.show()

## Section 3.3-3.5 — Triple Barrier Labeling

A more realistic labeling method. For each CUSUM event, three barriers are set:
- **Upper barrier (+1):** profit target — price rises by `trgt × ptSl[0]`
- **Lower barrier (-1):** stop loss — price falls by `trgt × ptSl[1]`  
- **Vertical barrier (0):** time limit — neither barrier hit within `num_days`

Whichever barrier is hit **first** determines the label. This accounts for
the path the price took, not just where it ended up.

In [ ]:
# Step 1: Add vertical barrier — max holding period of 3 days
t1 = labeling.add_vertical_barrier(close, events, num_days=3)
print(f"Vertical barriers set for {len(t1)} events")

# Step 2: Get events with all three barriers
# pt_sl = [1, 1]: symmetric barriers — profit target = stop loss = 1x daily vol
# min_ret = 0.005: skip events where volatility is below 0.5%
tb_events = labeling.get_events(
    close   = close,
    t_events= events,
    pt_sl   = [1, 1],
    trgt    = daily_vol,
    min_ret = 0.005,
    t1      = t1
)

print(f"Triple barrier events: {len(tb_events)}")
tb_events.head()

In [ ]:
# Step 3: Assign labels based on which barrier was hit first
tb_labels = labeling.get_bins(tb_events, close)

print(f"Triple barrier labels: {len(tb_labels)}")
print(f"Label distribution:")
print(tb_labels['bin'].value_counts().sort_index())
tb_labels.head(10)

### Triple Barrier — Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Triple Barrier Labeling (ptSl=[1,1], num_days=3)", fontsize=12)

# Bar chart of label counts
counts = tb_labels['bin'].value_counts().sort_index()
axes[0].bar(['-1 (Loss)', '0 (Neutral)', '+1 (Win)'],
            [counts.get(-1, 0), counts.get(0, 0), counts.get(1, 0)],
            color=['red', 'grey', 'green'])
axes[0].set_title("Label Counts")
axes[0].set_ylabel("Number of observations")

# Labels over time
axes[1].scatter(tb_labels.index, tb_labels['bin'].values,
                c=tb_labels['bin'].map({-1: 'red', 0: 'grey', 1: 'green'}),
                alpha=0.6, s=20)
axes[1].set_title("Labels Over Time")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Label")
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['-1 (Loss)', '0 (Neutral)', '+1 (Win)'])

axes[1].legend(handles=[green_patch, grey_patch, red_patch])
plt.tight_layout()
plt.show()

## Comparison — Fixed-Time Horizon vs Triple Barrier

The key difference between the two methods:

| | Fixed-Time Horizon | Triple Barrier |
|---|---|---|
| **What it measures** | Return after exactly h bars | Which barrier was hit first |
| **Accounts for path** | ❌ No | ✅ Yes |
| **Stop loss** | ❌ No | ✅ Yes |
| **Profit target** | ❌ No | ✅ Yes |
| **Time limit** | Fixed (always h bars) | Flexible (up to num_days) |
| **Neutral label** | When return < threshold | When time runs out |

The triple barrier method produces more realistic labels because it simulates
how a real trader with defined risk management would experience each entry.

In [ ]:
# Compare label distributions side by side
fth_counts = fth_labels.value_counts().sort_index()
tb_counts  = tb_labels['bin'].value_counts().sort_index()

comparison = pd.DataFrame({
    'Fixed-Time Horizon': fth_counts,
    'Triple Barrier':     tb_counts
}).fillna(0).astype(int)

comparison.index = ['-1 (Loss)', '0 (Neutral)', '+1 (Win)']
print("Label distribution comparison:")
print(comparison)
print()
print(f"Fixed-Time Horizon — % wins: {fth_counts.get(1,0)/len(fth_labels)*100:.1f}%")
print(f"Triple Barrier     — % wins: {tb_counts.get(1,0)/len(tb_labels)*100:.1f}%")

## Section 3.6-3.7 — Meta-Labeling

Meta-labeling splits the trading problem into two stages:

1. **Primary model** — decides the *side* of each trade (+1 = long, -1 = short). Here we use a simple rule: if the CUSUM event was triggered by upward drift, go long; if downward drift, go short.
2. **Meta-model** — decides *whether to act* on the primary signal. Labels are now binary: **1 = primary model was right**, **0 = primary model was wrong**.

This turns the problem into binary classification, which ML models handle much better than three-class problems.

### Step 1 — Primary Model Side Predictions

In a real implementation the primary model would be a trained classifier.
Here we simulate it using a simple momentum rule:
- If the close price at the event date is above the 20-bar moving average → long (+1)
- Otherwise → short (-1)

This is intentionally simple — the meta-model's job is to filter out the bad signals.

In [ ]:
# Compute 20-bar moving average on dollar bar closes
ma20 = close.rolling(20).mean()

# Primary model: long if price above MA, short if below
# Align to CUSUM event dates only
primary_side = pd.Series(
    np.where(close.reindex(events, method='bfill') > ma20.reindex(events, method='bfill'), 1, -1),
    index=events
)

print("Primary model side predictions:")
print(primary_side.value_counts())
print(f"\n{(primary_side == 1).sum()} long signals, {(primary_side == -1).sum()} short signals")

### Step 2 — Get Events with Side Information

We pass the primary model's side predictions into `get_events_meta()`.
The barriers now follow the predicted direction — profit target is in the
direction the primary model predicted, stop loss is against it.

In [ ]:
# Add vertical barrier
t1 = labeling.add_vertical_barrier(close, events, num_days=3)

# Get events with primary model side
meta_events = labeling.get_events_meta(
    close    = close,
    t_events = events,
    pt_sl    = [1, 1],
    trgt     = daily_vol,
    min_ret  = 0.005,
    t1       = t1,
    side     = primary_side
)

print(f"Meta-labeling events: {len(meta_events)}")
print(f"Side distribution in events:")
print(meta_events['side'].value_counts())
meta_events.head()

### Step 3 — Assign Meta-Labels

`get_bins_meta()` assigns binary labels:
- **1** = the primary model was correct (bet paid off)
- **0** = the primary model was wrong (bet lost)

This is what the meta-model will be trained to predict.

In [ ]:
# Assign meta-labels
meta_labels = labeling.get_bins_meta(meta_events, close)

print(f"Meta-labels: {len(meta_labels)}")
print(f"\nLabel distribution:")
print(meta_labels['bin'].value_counts().sort_index())
print(f"\nPrimary model accuracy: {meta_labels['bin'].mean()*100:.1f}%")
meta_labels.head(10)

### Meta-Labeling — Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Meta-Labeling: Was the Primary Model Right?", fontsize=12)

# Bar chart
counts = meta_labels['bin'].value_counts().sort_index()
axes[0].bar(['0 (Wrong)', '1 (Right)'],
            [counts.get(0, 0), counts.get(1, 0)],
            color=['red', 'green'])
axes[0].set_title("Primary Model Accuracy")
axes[0].set_ylabel("Number of observations")

# Returns by correctness
correct   = meta_labels[meta_labels['bin'] == 1]['ret']
incorrect = meta_labels[meta_labels['bin'] == 0]['ret']

axes[1].hist(correct * 100,   bins=15, alpha=0.6, color='green', label='Correct (bin=1)')
axes[1].hist(incorrect * 100, bins=15, alpha=0.6, color='red',   label='Wrong (bin=0)')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title("Return Distribution by Label")
axes[1].set_xlabel("Return (%)")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 3.9 — Dropping Unnecessary Labels

Some ML models perform poorly when label classes are heavily imbalanced.
`drop_labels()` recursively removes observations with extremely rare labels
until all remaining labels appear in at least `min_pct` of cases.

This is more relevant for three-class problems (triple barrier without meta-labeling)
where one of -1, 0, +1 might be very rare.

In [ ]:
# Apply drop_labels to the triple barrier output
# Using min_pct=0.10: drop any label appearing in less than 10% of cases
tb_labels_clean = labeling.drop_labels(tb_labels.copy(), min_pct=0.10)

print(f"Before drop_labels: {len(tb_labels)} observations")
print(f"After drop_labels:  {len(tb_labels_clean)} observations")
print(f"\nRemaining label distribution:")
print(tb_labels_clean['bin'].value_counts().sort_index())

## Summary — All Three Labeling Methods

| Method | Labels | What it measures |
|--------|--------|-----------------|
| **Fixed-Time Horizon** | -1, 0, +1 | Price direction after h bars |
| **Triple Barrier** | -1, 0, +1 | Which barrier was hit first |
| **Meta-Labeling** | 0, 1 | Was the primary model correct? |

The three methods build on each other:
1. Fixed-time horizon is the baseline — simple but ignores path and risk
2. Triple barrier improves on it by accounting for stop losses and profit targets
3. Meta-labeling uses triple barrier labels to train a second model that filters primary signals

In practice you would use all three together: triple barrier to generate labels,
meta-labeling to train a filter, and drop_labels to handle class imbalance.